# Nuclei instance segmentation on PanNuke — a method comparison

Instance segmentation of nuclei demands three things at once: **finding** each
nucleus, **naming** its type, and delineating its **boundary**. A single score
conflates them, so here we separate the axes and read every method against a
ground-truth-prompted **oracle** that fixes the achievable segmentation quality.
A method's distance from the oracle is then attributable to *detection* rather
than *masking*, and an optional post-hoc classifier isolates *naming* from both.

| method | how it localises nuclei | family |
|---|---|---|
| **oracle — GT boxes → SAM2** | ground-truth boxes | upper bound (segmentation ceiling) |
| **OWLv2 → SAM2** | open-vocabulary detector emits boxes | open-vocabulary detection |
| **LocateAnything-3B → SAM2** | grounding VLM emits boxes | text grounding (NVIDIA) |
| **SAM3 — text concept** | promptable concept segmentation, end to end | concept segmentation |

Every box method shares one **SAM2** backbone for masks, so the comparison isolates
how each model *points at* nuclei rather than how it draws them; SAM3 is end to end.
An optional **UNI2-h + MLP** classifier relabels each predicted nucleus from its
pixels, separating mask quality from the difficulty of subtyping.

> **`N_IMAGES`** and **`SEED`** (in the configuration) set how many random patches
> are shown and which draw. Runs on a single GPU; the LocateAnything weights are
> released for academic / non-profit research only.


## 1 — Bootstrap
We install a **recent transformers (5.x)** so `Sam3Model` is available. **Enable GPU + Internet** (Settings → Accelerator: GPU, Internet: On). If a previous attempt half-installed the package, **restart the kernel first** (Run → Restart & clear outputs).

- **SAM3 (`facebook/sam3`) is a gated model.** Request access on its HF page, then add a Kaggle Secret **`HF_TOKEN`** (Add-ons → Secrets) — this cell logs in with it so SAM3 can download.
- **LocateAnything** officially pins `transformers==4.57.1` and does **not** run on transformers 5.x; the run cell skips it here. Use the dedicated `locate_anything_pannuke_kaggle` notebook (pinned 4.57.1) for LA.

In [ ]:
import os, sys, subprocess
IN_KAGGLE = os.path.exists("/kaggle")
REPO = "git+https://github.com/marjanstoimchev/vlm-medseg.git@main"

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if IN_KAGGLE:
    # transformers 5.x carries Sam3Model (SAM3). LocateAnything pins 4.57.1 and breaks
    # on 5.x, so it is skipped in the run cell (use the dedicated LA notebook instead).
    pip("transformers==5.8.0", "decord==0.6.0", "lmdb==1.7.5", "peft", "accelerate", "einops", "timm")
    # Clean rebuild of our package only: a stale/cached copy from an earlier run can be
    # missing subpackages; --force-reinstall --no-cache-dir guarantees a fresh build,
    # --no-deps leaves Kaggle's preinstalled torch/transformers untouched.
    pip("--no-cache-dir", "--force-reinstall", "--no-deps", REPO)
else:
    pip("-e", "..")

# HF auth for the gated SAM3 repo (facebook/sam3). On Kaggle add a Secret named HF_TOKEN;
# locally a cached `huggingface-cli login` already works.
hf_token = os.environ.get("HF_TOKEN")
if IN_KAGGLE and not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF authenticated -- gated models (SAM3) enabled")
else:
    print("no HF_TOKEN -- SAM3 will be skipped; request access at hf.co/facebook/sam3 and add the secret")

from vlm_medseg.data import get_dataset  # smoke test: fail here, not 5 cells later
print("bootstrap done")

## 2 — Configuration
Everything per-method lives here:
- **`N_IMAGES` / `SEED`** — how many random patches and which draw.
- **Thresholds** — `OWL_THRESH`, `SAM3_THRESH` (Kaggle default **0.15**), `SAM3_MASK_THRESH`, `INPUT_SHORT`.
- **`SAHI_METHODS`** — which box detectors to tile (e.g. `{"owlv2"}`, `{"owlv2", "la"}`, or `set()` for none); SAM3 is end-to-end and ignores it.
- **Classifier** — `USE_CLASSIFIER` + `CLASSIFIER_DIR` (the attached UNI2-h Model). When on, it overrides each predicted nucleus class from its pixels for every method **except the oracle**.

In [ ]:
import gc, os, glob, numpy as np, torch
import matplotlib.pyplot as plt
from vlm_medseg.viz import set_paper_style
set_paper_style()                      # clean, consistent figure styling for every plot below

# -- dataset / sampling --
DATASET     = "pannuke"
FOLD        = "fold1"
N_IMAGES    = 6            # random patches to compare (raise for a bigger gallery)
SEED        = 0            # change to draw a different random set
CLASS_AWARE = True         # per-class prompts -> class-coloured overlays
IOU_THRESH  = 0.5

# -- per-method knobs --
INPUT_SHORT      = 1024    # upscale short side so the VLMs can see tiny nuclei
OWL_THRESH       = 0.1     # OWLv2 detection threshold
SAM3_THRESH      = 0.15    # SAM3 detection threshold
SAM3_MASK_THRESH = 0.5     # SAM3 mask binarisation threshold

# -- which methods to run (LA needs transformers 4.57 -> skipped on 5.x) --
RUN_OWLV2, RUN_LA, RUN_SAM3 = True, True, True

# -- SAHI tiling, box detectors only (SAM3 ignores it) --
SAHI_METHODS = {"owlv2"}          # e.g. set() to disable, or {"owlv2", "la"}
SAHI_TILE, SAHI_OVERLAP = 128, 0.25

# -- post-hoc nucleus classifier: overrides predicted class from pixels (UNI2-h + MLP) --
# Attach the Kaggle Model and point CLASSIFIER_DIR at it; the .joblib is auto-found.
# Its encoder (UNI2-h) is gated -> the same HF_TOKEN that unlocks SAM3 covers it.
USE_CLASSIFIER = True
CLASSIFIER_DIR = "/kaggle/input/models/marjan1111/uni2-h-mlp-classifier/scikitlearn/default/1"

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
DEVICE = "cuda" if IN_KAGGLE else "auto"
print("comparing", N_IMAGES, "patches from", f"{DATASET}/{FOLD}", "· seed", SEED,
      "· SAHI:", SAHI_METHODS or "off", "· classifier:", USE_CLASSIFIER)

## 3 — Sample patches
A uniform-random draw (not stratified), so the panel reflects a typical field of view rather than a curated one — change `SEED` to resample. Each patch shows its ground-truth nuclei: the **fill** encodes class and the **yellow outline** marks each boundary; the title gives tissue type and nucleus count.

In [ ]:
import random
from vlm_medseg.data import get_dataset
from vlm_medseg.viz import overlay_gt, class_legend_handles

ds = get_dataset(DATASET, fold=FOLD); spec = ds.spec
idx = random.Random(SEED).sample(range(len(ds)), N_IMAGES)
samples = [ds.decode(i) for i in idx]
print("nuclei per patch:", [s.num_instances for s in samples])

cols = min(N_IMAGES, 6)
rows = (N_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(2.7 * cols, 2.9 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, s in zip(axes.flat, samples):
    ax.imshow(overlay_gt(s, spec=spec))
    ax.set_title(f"{spec.group_name(s.group)} · {s.num_instances} nuclei", fontsize=9)
fig.legend(handles=class_legend_handles(spec), loc="lower center", ncol=spec.num_classes,
           bbox_to_anchor=(0.5, 0.0), fontsize=9, title_fontsize=9,
           title="nucleus class — fill colour   ·   yellow outline = boundary")
fig.suptitle("Ground-truth nuclei — the targets", y=1.0)
fig.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

## 4 — Run each method
SAM2 is loaded once and shared; each detector/segmenter is loaded, run over the patches, then freed (so it fits a 16 GB GPU). The oracle uses ground-truth boxes. Detectors in **`SAHI_METHODS`** are tiled first, and if the **classifier** loaded it relabels every method's nuclei **except the oracle**.

> **Version note.** This run uses transformers 5.x. **SAM3** needs HF auth — add an `HF_TOKEN` secret (see Bootstrap), or it is skipped as a gated repo. **LocateAnything** pins `transformers==4.57.1` and does **not** run on 5.x, so it is skipped here (set `RUN_LA=False` to silence the attempt); use the dedicated `locate_anything_pannuke_kaggle` notebook for LA. The comparison still renders for whatever ran.

In [ ]:
from vlm_medseg.segment.sam2 import Sam2Masker
from vlm_medseg.detect.oracle import OracleBoxDetector
from vlm_medseg.detect.sahi import SahiDetector
from vlm_medseg.pipeline.run import run_condition, run_segmenter_condition

# Optional post-hoc classifier: relabels each predicted nucleus from its pixels.
classifier = None
if USE_CLASSIFIER:
    from vlm_medseg.classify.linear_probe import NucleusClassifier
    hits = sorted(glob.glob(os.path.join(CLASSIFIER_DIR, "*.joblib")))
    if hits:
        classifier = NucleusClassifier.load(hits[0], device=DEVICE)
        print("classifier:", os.path.basename(hits[0]), "->", classifier.class_names)
    else:
        print("USE_CLASSIFIER set but no .joblib under", CLASSIFIER_DIR, "-- continuing without it")

def sahi(det, key):   # tile a box detector only if it is listed in SAHI_METHODS
    return SahiDetector(det, tile=SAHI_TILE, overlap=SAHI_OVERLAP) if key in SAHI_METHODS else det

masker = Sam2Masker(device=DEVICE)
runs = {}

# Oracle is the segmentation ceiling: ground-truth boxes AND classes -> never classified.
runs["oracle"] = run_condition(OracleBoxDetector(class_aware=CLASS_AWARE, spec=spec),
                               masker, samples, spec=spec, iou_thresh=IOU_THRESH)

# Each optional method is isolated: a failure skips just that method (with a note), so the
# rest still renders. Detectors in SAHI_METHODS are tiled; the classifier (if loaded)
# overrides predicted classes for every method below.
if RUN_OWLV2:
    try:
        from vlm_medseg.detect.owlv2 import Owlv2Detector
        owl = sahi(Owlv2Detector(device=DEVICE, class_aware=CLASS_AWARE, spec=spec, threshold=OWL_THRESH), "owlv2")
        runs["owlv2"] = run_condition(owl, masker, samples, spec=spec, iou_thresh=IOU_THRESH, classifier=classifier)
        del owl; free()
    except Exception as e:
        print("owlv2 skipped:", type(e).__name__, e)

if RUN_LA:
    try:
        from vlm_medseg.detect.locate_anything import LocateAnythingDetector
        la = sahi(LocateAnythingDetector(device="cuda", class_aware=CLASS_AWARE, spec=spec, input_short_size=INPUT_SHORT), "la")
        runs["la"] = run_condition(la, masker, samples, spec=spec, iou_thresh=IOU_THRESH, classifier=classifier)
        del la; free()
    except Exception as e:
        print("locate-anything skipped:", type(e).__name__, e)
        print("  LocateAnything needs transformers==4.57.1 and does not run on 5.x --")
        print("  use the dedicated locate_anything_pannuke_kaggle notebook for LA.")

if RUN_SAM3:
    try:
        from vlm_medseg.segment.sam3 import Sam3TextSegmenter
        sam3 = Sam3TextSegmenter(device=DEVICE, spec=spec, class_aware=CLASS_AWARE,
                                 threshold=SAM3_THRESH, mask_threshold=SAM3_MASK_THRESH)
        runs["sam3"] = run_segmenter_condition(sam3, samples, spec=spec, iou_thresh=IOU_THRESH, classifier=classifier)
        del sam3; free()
    except ImportError as e:
        print("SAM3 skipped -- this transformers build has no Sam3Model:", e)
        print("  The bootstrap installs transformers 5.x for SAM3; if you see this, repin")
        print("  transformers so Sam3Model is importable, then restart the kernel.")
    except Exception as e:
        print("SAM3 skipped:", type(e).__name__, e)
        if "gated" in str(e).lower() or "401" in str(e):
            print("  facebook/sam3 is gated: request access at hf.co/facebook/sam3 and add an")
            print("  HF_TOKEN Kaggle Secret (the bootstrap logs in with it), then restart.")

del masker
if classifier is not None: del classifier
free()
print("ran:", list(runs))

## 5 — Qualitative comparison
One row per patch: **H&E · ground truth · each method**. The fill encodes the predicted class and the **yellow outline** marks each boundary, so the figure reads like a contact sheet — missed nuclei, background bleed, merged neighbours, and class confusion are all apparent against the ground-truth column, which is the target each method is reproducing.

In [ ]:
from vlm_medseg.viz import gallery
pred_by_cond = {name: [r.instances for r in out["results"]] for name, out in runs.items()}
fig = gallery(samples, pred_by_cond, spec=spec, max_rows=N_IMAGES)
fig.suptitle("H&E | GT | " + " | ".join(runs), y=1.005, fontsize=12); plt.show()

## 6 — Quantitative comparison
The panoptic metrics are read as a **decomposition**, not a single figure of merit. **PQ** is overall panoptic quality; **AJI** and **Dice** summarise pixel agreement; **matched IoU** is mask quality on the nuclei a method actually recovers; **detection recall / F1** and the **predicted-to-true count ratio** capture how completely it finds them. Reading matched IoU against recall separates *how well a method segments what it finds* from *how much it finds* — and the oracle row bounds the former.

In [ ]:
import pandas as pd
def row(name, out):
    s = out["summary"]; d = s["detection_pooled"]
    r = {"method": name, "PQ": s["binary_pq"], "AJI": s["aji"], "Dice": s["dice"],
         "matchedIoU": s["matched_iou"], "det_R": d["recall"], "det_F1": d["f1"],
         "pred/gt": s["counting"]["total_pred"] / max(1, s["counting"]["total_gt"])}
    if "mpq" in s: r["mPQ"] = s["mpq"]
    return r
table = pd.DataFrame([row(k, v) for k, v in runs.items()]).set_index("method")
display(table.round(3))

from vlm_medseg.viz import plot_metric_bars
plot_metric_bars({k: v["summary"] for k, v in runs.items()}).show()

## 7 — A single field, up close
The most densely populated patch in the draw, each method beside the ground truth, to inspect boundary adherence and class assignment at the level of individual nuclei.

In [ ]:
from vlm_medseg.viz import comparison_panel
busiest = int(np.argmax([s.num_instances for s in samples]))
s = samples[busiest]
panel = {name: out["results"][busiest].instances for name, out in runs.items()}
fig = comparison_panel(s, panel, spec=spec, figsize_scale=3.4)
fig.suptitle(f"{spec.group_name(s.group)} · {s.num_instances} nuclei", y=1.03); plt.show()

## 8 — Reading the comparison

- **The oracle is the segmentation ceiling.** Given perfect boxes, SAM2 delineates nuclei well, so every other method is best read as a *fraction* of the oracle rather than in absolute terms; the distance to it localises where a method fails.
- **Detection versus segmentation.** When a method's *matched IoU* tracks the oracle while its *PQ* and *recall* trail behind, the masks it does produce are sound and the loss is one of **detection** — nuclei that are never found cannot be segmented. The count ratio corroborates this: pronounced under- or over-counting points to localisation rather than delineation.
- **Specialised versus generic localisers.** Contrasting the grounding VLM with the generic open-vocabulary detector shows whether task-specific grounding translates into denser, more complete nucleus localisation on histology, or whether a general detector already suffices.
- **Detector-free segmentation.** SAM3 reaches nuclei without an explicit detector, but its text concepts separate subtypes only weakly, so raw class assignment tends to collapse toward a single label — visible as near-uniform fill. This is precisely the gap the post-hoc classifier is meant to close.
- **Naming as a separate axis.** With the classifier enabled, the fill colours reflect a pixel-level subtype prediction rather than the prompt label; toggling it isolates how much of the class confusion is a *naming* problem solvable after segmentation versus a *segmentation* problem in the masks themselves.
- **Stability of the reading.** Raising **`N_IMAGES`** widens the sample and changing **`SEED`** resamples it; the qualitative ordering of methods should persist across draws, which is what makes the comparison trustworthy beyond a single figure.
